In [1]:
import os
import sys
import time
import inspect
import contextlib
import pandas as pd
import numpy as np
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import math

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

In [2]:
def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

BASE_SEED = 42
seed_everything(BASE_SEED)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Загрузка датасетов

In [ ]:
TRAIN_PATH = "../datasets/train.parquet"
VALID_PATH = "../datasets/valid.parquet"

train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)

META_COLS = ["seq_ix", "step_in_seq", "need_prediction"]
FEATURE_COLS = train_df.columns[3:35].tolist()   # 32 feats
TARGET_COLS  = train_df.columns[35:37].tolist()  # 2 targets

print("train shape:", train_df.shape)
print("valid shape:", valid_df.shape)
print("features:", len(FEATURE_COLS), "targets:", len(TARGET_COLS))
print("target cols:", TARGET_COLS)

train shape: (10721000, 37)
valid shape: (1444000, 37)
features: 32 targets: 2
target cols: ['t0', 't1']


In [ ]:
train_seq_ids = train_df["seq_ix"].unique()
valid_seq_ids = valid_df["seq_ix"].unique()

print("num train seq:", len(train_seq_ids))
print("num valid seq:", len(valid_seq_ids))

num train seq: 10721
num valid seq: 1444


## Подготовка данных к обучению

### Собираем seqs

In [ ]:
def build_sequences(df, feature_cols, target_cols):
    seq_ids = np.sort(df["seq_ix"].unique())
    n_seq = len(seq_ids)
    
    X_seqs = np.empty((n_seq, 1000, len(feature_cols)), dtype=np.float32)
    Y_seqs = np.empty((n_seq, 1000, len(target_cols)), dtype=np.float32)
    M_seqs = np.empty((n_seq, 1000), dtype=bool)

    grouped = df.groupby("seq_ix", sort=False)
    
    for i, sid in enumerate(seq_ids):
        g = grouped.get_group(sid).sort_values("step_in_seq")
        
        # (1000, 32)
        X_seqs[i] = g[feature_cols].to_numpy(dtype=np.float32, copy=False)
        # (1000, 2)
        Y_seqs[i] = g[target_cols].to_numpy(dtype=np.float32, copy=False)
        # (1000,)
        M_seqs[i] = g["need_prediction"].to_numpy(dtype=bool, copy=False)

    return X_seqs, Y_seqs, M_seqs, seq_ids

In [ ]:
X_train, Y_train, M_train, train_seq_ids = build_sequences(train_df, FEATURE_COLS, TARGET_COLS)
X_valid, Y_valid, M_valid, valid_seq_ids = build_sequences(valid_df, FEATURE_COLS, TARGET_COLS)

print("X_train:", X_train.shape, X_train.dtype)
print("Y_train:", Y_train.shape, Y_train.dtype)
print("M_train:", M_train.shape, M_train.dtype)
print("")
print("X_valid:", X_valid.shape, X_valid.dtype)
print("Y_valid:", Y_valid.shape, Y_valid.dtype)
print("M_valid:", M_valid.shape, M_valid.dtype)

X_train: (10721, 1000, 32) float32
Y_train: (10721, 1000, 2) float32
M_train: (10721, 1000) bool

X_valid: (1444, 1000, 32) float32
Y_valid: (1444, 1000, 2) float32
M_valid: (1444, 1000) bool


### Собираем Dataloaders

In [ ]:
class SequenceStoreDataset(Dataset):
    def __init__(self, X_seqs, Y_seqs, M_seqs):
        self.X = X_seqs
        self.Y = Y_seqs
        self.M = M_seqs

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.M[idx]

class WindowCollator:
    def __init__(self, window=100, K=8, scored_only=True, base_seed=42):
        self.window = window
        self.K = K
        self.scored_only = scored_only
        self.base_seed = int(base_seed)
        self.epoch = 0
        self.call_id = 0

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)
        self.call_id = 0

    def __call__(self, batch):
        seed = self.base_seed + self.epoch * 1_000_000 + self.call_id
        self.call_id += 1
        rng = np.random.default_rng(seed)

        X_list, y_list = [], []
        window = self.window
        K = self.K

        for X_seq, Y_seq, M_seq in batch:
            if self.scored_only:
                t_candidates = np.where(M_seq)[0]
            else:
                t_candidates = np.arange(len(M_seq))

            t_candidates = t_candidates[t_candidates >= (window - 1)]
            if len(t_candidates) == 0:
                continue

            replace = len(t_candidates) < K
            t_pick = rng.choice(t_candidates, size=K, replace=replace)

            for t in t_pick:
                start = t - (window - 1)
                end = t + 1
                X_list.append(X_seq[start:end, :])
                y_list.append(Y_seq[t, :])

        Xb = torch.tensor(np.stack(X_list), dtype=torch.float32)
        yb = torch.tensor(np.stack(y_list), dtype=torch.float32)
        return Xb, yb

In [ ]:
def make_window_collate(window=100, K=8, scored_only=True, seed=None):
    rng = np.random.default_rng(seed)

    def collate(batch):
        X_list, y_list = [], []

        for X_seq, Y_seq, M_seq in batch:
            if scored_only:
                t_candidates = np.where(M_seq)[0]         
            else:
                t_candidates = np.arange(len(M_seq))

            t_candidates = t_candidates[t_candidates >= (window - 1)] 
            
            if len(t_candidates) == 0:
                continue

            replace = len(t_candidates) < K
            t_pick = rng.choice(t_candidates, size=K, replace=replace)

            for t in t_pick:
                start = t - (window - 1)
                end = t + 1
                X_list.append(X_seq[start:end, :])   # (100,32)
                y_list.append(Y_seq[t, :])           # (2,)

        Xb = torch.tensor(np.stack(X_list), dtype=torch.float32)  # (B_seq*K,100,32)
        yb = torch.tensor(np.stack(y_list), dtype=torch.float32)  # (B_seq*K,2)
        return Xb, yb

    return collate

In [ ]:
WINDOW = 100
B_SEQ = 72     # sequences in batch
K = 256        # windows in sequence

train_collator = WindowCollator(window=100, K=K, scored_only=True, base_seed=BASE_SEED)
valid_collator = WindowCollator(window=WINDOW, K=K, scored_only=True, base_seed=123)

g = torch.Generator()
g.manual_seed(BASE_SEED)

train_seq_ds = SequenceStoreDataset(X_train, Y_train, M_train)
valid_seq_ds = SequenceStoreDataset(X_valid, Y_valid, M_valid)

train_loader = DataLoader(
    train_seq_ds,
    batch_size=B_SEQ,
    shuffle=True,
    num_workers=0,     
    generator=g,       
    collate_fn=train_collator,
    drop_last=True,
)

valid_loader = DataLoader(
    valid_seq_ds,
    batch_size=B_SEQ,
    shuffle=False,
    num_workers=0,
    collate_fn=valid_collator,
    drop_last=False,
)

xb, yb = next(iter(train_loader))
print("X batch:", xb.shape, xb.dtype)   # (B_SEQ*K, 100, 32)
print("y batch:", yb.shape, yb.dtype)   # (B_SEQ*K, 2)

X batch: torch.Size([18432, 100, 32]) torch.float32
y batch: torch.Size([18432, 2]) torch.float32


## Обучение

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 100):
        super().__init__()
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)  # (T, D)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (T,1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )  # (D/2,)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, T, D)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,D)
        T = x.size(1)
        return x + self.pe[:, :T, :]


class ConvDilatedBlock(nn.Module):
    def __init__(self, ch: int, kernel_size: int = 3, dilation: int = 1, dropout: float = 0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2  
        self.conv1 = nn.Conv1d(ch, ch, kernel_size=kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(ch, ch, kernel_size=kernel_size, padding=pad, dilation=dilation)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.norm = nn.GroupNorm(num_groups=1, num_channels=ch)  

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        r = x
        x = self.drop(self.act(self.conv1(x)))
        x = self.drop(self.act(self.conv2(x)))
        x = self.norm(x + r)
        return x


class ConvTransformerTwoHeadRegressor(nn.Module):
    def __init__(
        self,
        input_dim: int = 32,
        window: int = 100,
        embed_dim: int = 64,          
        conv_blocks: int = 2,         
        conv_kernel: int = 3,
        conv_dropout: float = 0.1,
        attn_layers: int = 1,         
        attn_heads: int = 4,          
        ffn_mult: int = 4,            
        attn_dropout: float = 0.0,    
        head_dim_0: int = 64,
        head_dim_1: int = 128,
        head_dropout: float = 0.1,
    ):
        super().__init__()
        assert embed_dim % attn_heads == 0, "embed_dim must be divisible by attn_heads"

        self.input_proj = nn.Linear(input_dim, embed_dim)

        self.conv_in_norm = nn.LayerNorm(embed_dim)
        self.conv = nn.Sequential(*[
            ConvDilatedBlock(
                ch=embed_dim,
                kernel_size=conv_kernel,
                dilation=(2 ** i),     
                dropout=conv_dropout,
            )
            for i in range(conv_blocks)
        ])

        self.pos = SinusoidalPositionalEncoding(embed_dim, max_len=window)

        if attn_layers > 0:
            enc_layer = nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=attn_heads,
                dim_feedforward=embed_dim * ffn_mult,
                dropout=attn_dropout,
                batch_first=True,
                activation="gelu",
                norm_first=True,
            )
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=attn_layers)
        else:
            self.transformer = None

        self.final_norm = nn.LayerNorm(embed_dim)

        self.head_t0 = nn.Sequential(
            nn.Linear(embed_dim, head_dim_0),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_0, 1),
        )
        self.head_t1 = nn.Sequential(
            nn.Linear(embed_dim, head_dim_1),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_1, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,32)
        x = self.input_proj(x)          # (B,T,E)
        x = self.conv_in_norm(x)

        # conv: (B,E,T)
        z = x.transpose(1, 2)           # (B,E,T)
        z = self.conv(z)                # (B,E,T)
        z = z.transpose(1, 2)           # (B,T,E)

        z = self.pos(z)                 # (B,T,E)
        if self.transformer is not None:
            z = self.transformer(z)     # (B,T,E)

        h = self.final_norm(z[:, -1, :])  # (B,E)

        y0 = self.head_t0(h)            # (B,1)
        y1 = self.head_t1(h)            # (B,1)
        out = torch.cat([y0, y1], dim=1)  # (B,2)
        return out


In [ ]:
def weighted_huber_loss(pred, y, delta=1.0, eps=1e-3):
    w = torch.abs(y).clamp_min(eps)  # (B,2)
    hub = F.smooth_l1_loss(pred, y, reduction="none", beta=delta)  # (B,2)
    return (w * hub).mean()

In [ ]:
def hybrid_weighted_huber_corr_loss(
    pred: torch.Tensor,
    y: torch.Tensor,
    delta: float = 1.0,
    eps: float = 1e-8,
    clip: float = 6.0,
    huber_w: float = 0.2,   
    corr_w: float = 0.8,    
) -> torch.Tensor:

    pred_c = clip * torch.tanh(pred / clip)

    w = torch.abs(y).clamp_min(eps)  # (B,2)

    hub = F.smooth_l1_loss(pred_c, y, reduction="none", beta=delta)  # (B,2)
    huber_loss = (w * hub).mean()

    sum_w = w.sum(dim=0).clamp_min(eps)  # (2,)

    mean_y = (w * y).sum(dim=0) / sum_w          # (2,)
    mean_p = (w * pred_c).sum(dim=0) / sum_w     # (2,)

    dev_y = y - mean_y
    dev_p = pred_c - mean_p

    cov = (w * dev_y * dev_p).sum(dim=0) / sum_w           # (2,)
    var_y = (w * dev_y.pow(2)).sum(dim=0) / sum_w          # (2,)
    var_p = (w * dev_p.pow(2)).sum(dim=0) / sum_w          # (2,)

    corr = cov / (torch.sqrt(var_y * var_p).clamp_min(eps))  # (2,)
    corr = torch.clamp(corr, -1.0, 1.0)

    corr_loss = (1.0 - corr).mean()  

    return huber_w * huber_loss + corr_w * corr_loss

In [ ]:
def weighted_npcc_loss(
    pred: torch.Tensor,
    y: torch.Tensor,
    eps: float = 1e-8,
    clip: float = 6.0,
) -> torch.Tensor:
    pred_c = clip * torch.tanh(pred / clip)

    w = torch.abs(y).clamp_min(eps)          # (B,2)
    sum_w = w.sum(dim=0).clamp_min(eps)      # (2,)

    mean_y = (w * y).sum(dim=0) / sum_w
    mean_p = (w * pred_c).sum(dim=0) / sum_w

    dev_y = y - mean_y
    dev_p = pred_c - mean_p

    cov  = (w * dev_y * dev_p).sum(dim=0) / sum_w
    var_y = (w * dev_y.pow(2)).sum(dim=0) / sum_w
    var_p = (w * dev_p.pow(2)).sum(dim=0) / sum_w

    corr = cov / torch.sqrt((var_y * var_p).clamp_min(eps))
    corr = torch.clamp(corr, -1.0, 1.0)

    return (1.0 - corr).mean()

In [ ]:
model = ConvTransformerTwoHeadRegressor(
    input_dim=32,
    window=100,
    embed_dim=96,       
    conv_blocks=3,      
    conv_kernel=5,      
    conv_dropout=0.2,
    attn_layers=1,      
    attn_heads=4,
    ffn_mult=4,
    attn_dropout=0.0,
    head_dim_0=64,
    head_dim_1=128,
    head_dropout=0.1,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)

/tmp/ipykernel_5949/3894754348.py:122: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=attn_layers)


In [ ]:
def weighted_pearson_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.clip(y_pred, -6.0, 6.0)

    weights = np.abs(y_true)
    weights = np.maximum(weights, 1e-8)

    sum_w = np.sum(weights)
    if sum_w == 0:
        return 0.0

    mean_true = np.sum(y_true * weights) / sum_w
    mean_pred = np.sum(y_pred * weights) / sum_w

    dev_true = y_true - mean_true
    dev_pred = y_pred - mean_pred

    cov = np.sum(weights * dev_true * dev_pred) / sum_w
    var_true = np.sum(weights * dev_true**2) / sum_w
    var_pred = np.sum(weights * dev_pred**2) / sum_w

    if var_true <= 0 or var_pred <= 0:
        return 0.0

    return float(cov / (np.sqrt(var_true) * np.sqrt(var_pred)))


def weighted_pearson_2targets(y_true_2: np.ndarray, y_pred_2: np.ndarray) -> dict:
    t0 = weighted_pearson_np(y_true_2[:, 0], y_pred_2[:, 0])
    t1 = weighted_pearson_np(y_true_2[:, 1], y_pred_2[:, 1])
    return {"t0": t0, "t1": t1, "weighted_pearson": 0.5 * (t0 + t1)}

In [ ]:
@torch.no_grad()
def evaluate_full_valid(
    model,
    X_valid: np.ndarray,  # (N_seq, 1000, 32) float32
    Y_valid: np.ndarray,  # (N_seq, 1000, 2)  float32
    M_valid: np.ndarray,  # (N_seq, 1000)     bool
    device,
    window=100,
    eval_batch_size=512,
    loss_fn=None,                
    loss_kwargs=None,            
):
    model.eval()
    if loss_kwargs is None:
        loss_kwargs = {}

    preds_all = []
    y_all = []

    total_loss = 0.0
    total_batches = 0

    X_buf, Y_buf = [], []

    n_seq = X_valid.shape[0]

    pbar = tqdm(range(n_seq), desc="Valid inference (full)", leave=False)
    for s in pbar:
        ts = np.where(M_valid[s])[0]
        ts = ts[ts >= (window - 1)]  

        X_seq = X_valid[s]
        Y_seq = Y_valid[s]

        for t in ts:
            start = t - (window - 1)
            end = t + 1
            X_buf.append(X_seq[start:end, :])   # (window,32)
            Y_buf.append(Y_seq[t, :])           # (2,)

            if len(X_buf) >= eval_batch_size:
                xb = torch.tensor(np.stack(X_buf), dtype=torch.float32, device=device)
                yb = torch.tensor(np.stack(Y_buf), dtype=torch.float32, device=device)

                pred = model(xb)  # (B,2)

                if loss_fn is not None:
                    loss = loss_fn(pred, yb, **loss_kwargs)
                    total_loss += float(loss.detach().item())
                    total_batches += 1

                preds_all.append(pred.detach().cpu().numpy())
                y_all.append(yb.detach().cpu().numpy())

                X_buf, Y_buf = [], []

    if len(X_buf) > 0:
        xb = torch.tensor(np.stack(X_buf), dtype=torch.float32, device=device)
        yb = torch.tensor(np.stack(Y_buf), dtype=torch.float32, device=device)
        pred = model(xb)

        if loss_fn is not None:
            loss = loss_fn(pred, yb, **loss_kwargs)
            total_loss += float(loss.detach().item())
            total_batches += 1

        preds_all.append(pred.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    preds_all = np.concatenate(preds_all, axis=0)  # (N,2)
    y_all = np.concatenate(y_all, axis=0)          # (N,2)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    valid_loss = total_loss / max(total_batches, 1) if loss_fn is not None else float("nan")

    return valid_loss, pearson


@torch.no_grad()
def evaluate_fast_valid(
    model,
    valid_loader,
    device,
    loss_fn,
    loss_kwargs=None,        
):
    model.eval()
    if loss_kwargs is None:
        loss_kwargs = {}

    preds_all = []
    y_all = []

    loss_sum = 0.0
    iters = 0

    with tqdm(total=len(valid_loader), desc="Valid (fast)", unit="batch", leave=False) as pbar:
        for xb, yb in valid_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            pred = model(xb)  # (B,2)
            loss = loss_fn(pred, yb, **loss_kwargs)

            loss_sum += float(loss.detach().item())
            iters += 1

            preds_all.append(pred.detach().cpu().numpy())
            y_all.append(yb.detach().cpu().numpy())

            pbar.update(1)

    preds_all = np.concatenate(preds_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    valid_loss = loss_sum / max(iters, 1)
    return valid_loss, pearson


In [ ]:
@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout, sys.stderr = devnull, devnull
            yield
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr

def export_onnx_silent(model_cpu, onnx_path, window=100, input_dim=32, opset=18):
    model_cpu.eval()
    dummy = torch.zeros(1, window, input_dim, dtype=torch.float32)  

    export_kwargs = dict(
        input_names=["x"],
        output_names=["y"],
        opset_version=opset,
        dynamic_axes={"x": {0: "batch"}, "y": {0: "batch"}},
    )

    sig = inspect.signature(torch.onnx.export)
    if "dynamo" in sig.parameters:
        export_kwargs["dynamo"] = False

    with suppress_output():
        torch.onnx.export(model_cpu, dummy, onnx_path, **export_kwargs)

In [ ]:
def train_model(
    model_nn,
    train_loader,
    optimizer,
    num_epochs,
    device,
    X_valid, Y_valid, M_valid,
    loss_fn,                     
    loss_kwargs=None,             
    eval_batch_size=512,
    window=100,
    valid_loader=None,            
    use_full_valid=True,
    full_valid_every=0,           
    save_dir_pth="models_pth",
    save_dir_onnx="models_onnx",
    save_each_epoch=True,
    onnx_opset=18,
):
    if loss_kwargs is None:
        loss_kwargs = {}

    os.makedirs(save_dir_pth, exist_ok=True)
    os.makedirs(save_dir_onnx, exist_ok=True)

    model_nn = model_nn.to(device)

    for epoch in range(num_epochs):
        if hasattr(train_loader.collate_fn, "set_epoch"):
            train_loader.collate_fn.set_epoch(epoch)
        if valid_loader is not None and hasattr(valid_loader.collate_fn, "set_epoch"):
            valid_loader.collate_fn.set_epoch(epoch)

        start = time.time()

        # ---- TRAIN ----
        model_nn.train()
        train_loss_sum = 0.0
        train_iters = 0

        with tqdm(total=len(train_loader), desc=f"Epoch {epoch+1} - Train", unit="batch", leave=False) as pbar:
            for xb, yb in train_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                pred = model_nn(xb)                      # (B,2)
                loss = loss_fn(pred, yb, **loss_kwargs)  

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_nn.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss_sum += float(loss.detach().item())
                train_iters += 1
                pbar.update(1)

        train_loss_out = train_loss_sum / max(train_iters, 1)

        # ---- VALID ----
        do_full = False
        if use_full_valid:
            if full_valid_every == 0:
                do_full = True
            else:
                do_full = (epoch % (full_valid_every + 1) == 0)

        if do_full:
            valid_loss_out, pearson = evaluate_full_valid(
                model_nn,
                X_valid=X_valid,
                Y_valid=Y_valid,
                M_valid=M_valid,
                device=device,
                window=window,
                eval_batch_size=eval_batch_size,
                loss_fn=loss_fn,              
                loss_kwargs=loss_kwargs,      
            )
            valid_mode = "FULL"
        else:
            if valid_loader is None:
                valid_loss_out = float("nan")
                pearson = {"weighted_pearson": float("nan"), "t0": float("nan"), "t1": float("nan")}
                valid_mode = "N/A"
            else:
                valid_loss_out, pearson = evaluate_fast_valid(
                    model_nn,
                    valid_loader=valid_loader,
                    device=device,
                    loss_fn=loss_fn,          
                    loss_kwargs=loss_kwargs,  
                )
                valid_mode = "FAST"

        elapsed = time.time() - start

        tqdm.write(
            f"Epoch: {epoch+1:02d} | Time: {elapsed:.2f}s | "
            f"Train Loss: {train_loss_out:.6f} | "
            f"Valid({valid_mode}) Loss: {valid_loss_out:.6f} | "
            f"Valid WPearson: {pearson['weighted_pearson']:.6f}"
        )

        # ---- SAVE ----
        if save_each_epoch:
            pth_path = os.path.join(save_dir_pth, f"epoch_{epoch+1:02d}.pth")
            torch.save(model_nn.state_dict(), pth_path)

            onnx_path = os.path.join(save_dir_onnx, f"epoch_{epoch+1:02d}.onnx")
            model_cpu = model_nn.to("cpu")
            export_onnx_silent(model_cpu, onnx_path, window=window, input_dim=32, opset=onnx_opset)
            model_nn.to(device)

    return model_nn

In [21]:
loss_fn = hybrid_weighted_huber_corr_loss
loss_kwargs = {
    "delta": 1.0,
    "eps": 1e-8,
    "clip": 6.0,
    "huber_w": 0.1,
    "corr_w": 0.9,
}

model_trained = train_model(
    model_nn=model,
    train_loader=train_loader,
    optimizer=optimizer,
    num_epochs=250,
    device=device,
    X_valid=X_valid, Y_valid=Y_valid, M_valid=M_valid,
    loss_fn=loss_fn,
    loss_kwargs=loss_kwargs,
    valid_loader=valid_loader,
    use_full_valid=True,
    full_valid_every=0,
    save_dir_pth="models_6_pth",
    save_dir_onnx="models_6_onnx"
)

Epoch: 01 | Time: 78.07s | Train Loss: 0.898451 | Valid(FULL) Loss: 1.033199 | Valid WPearson: 0.118251


Epoch: 02 | Time: 77.05s | Train Loss: 0.847530 | Valid(FULL) Loss: 1.001897 | Valid WPearson: 0.145340


Epoch: 03 | Time: 77.54s | Train Loss: 0.827449 | Valid(FULL) Loss: 0.987329 | Valid WPearson: 0.159629


Epoch: 04 | Time: 77.43s | Train Loss: 0.816339 | Valid(FULL) Loss: 0.982306 | Valid WPearson: 0.169287


Epoch: 05 | Time: 78.10s | Train Loss: 0.804347 | Valid(FULL) Loss: 0.974781 | Valid WPearson: 0.178897


...

Epoch: 246 | Time: 77.60s | Train Loss: 0.614405 | Valid(FULL) Loss: 0.902927 | Valid WPearson: 0.213896


Epoch: 247 | Time: 77.70s | Train Loss: 0.614069 | Valid(FULL) Loss: 0.906039 | Valid WPearson: 0.208722


Epoch: 248 | Time: 78.11s | Train Loss: 0.611323 | Valid(FULL) Loss: 0.905190 | Valid WPearson: 0.209191


Epoch: 249 | Time: 77.56s | Train Loss: 0.608261 | Valid(FULL) Loss: 0.904965 | Valid WPearson: 0.212523


Epoch: 250 | Time: 77.53s | Train Loss: 0.613178 | Valid(FULL) Loss: 0.904875 | Valid WPearson: 0.208827
